# Using ServiceX

The code in this notebook is the code that is found in the [ServiceX 15 Minute Challenge](https://tryservicex.org/challenge/). Please see the challenge directly for more information and explination for each step and section of code.

For more general information on ServiceX and using it please see the [ServiceX User Guide](https://tryservicex.org/guide/).

## Setup

You may need to install these libraries:

`conda install --yes vector servicex`

Try running the code first, they should already be installed!

In [ ]:
from servicex import deliver, query, dataset

import awkward as ak
import uproot
import vector
vector.register_awkward()
import hist

import matplotlib.pyplot as plt

In [ ]:
# Specify dataset
file = "root://eos.cms.rcac.purdue.edu:1094//store/mc/RunIII2024Summer24NanoAODv15/VBFH-Hto2Mu_Par-M-125_TuneCP5_13p6TeV_powheg-pythia8/NANOAODSIM/150X_mcRun3_2024_realistic_v2-v2/100000/1d437865-3bfb-4d90-b596-c2f638d398f3.root"
nanoaod_dataset = dataset.FileList([file])

# Specify columns and cuts
uproot_raw_query = query.UprootRaw([{
        'treename':'Events',
        'filter_name':
            ["Muon_pt", "Muon_eta", "Muon_phi", "Muon_mass", 'Muon_mediumId', 'Muon_charge'],
        'cut': '(nMuon == 2) & all(Muon_mediumId & (Muon_pt > 20) & (abs(Muon_eta) < 2.4)) & (sum(Muon_charge) == 0)'
    }]
)

# Prepare ServiceX request
spec_uproot_raw = {
    'Sample': [{
        'Name': 'UprootRawExample',
        'Dataset': nanoaod_dataset,
        'Query': uproot_raw_query
    }]
}

In [ ]:
# Run ServiceX transformers 
results_uproot_raw=deliver(spec_uproot_raw)

In [ ]:
# Read output file
skimmed_file = results_uproot_raw["UprootRawExample"][0]
with uproot.open(skimmed_file) as f:
    arrays = f["Events"].arrays()

# Compute dimuon mass
muons = ak.zip({
    'pt':   arrays['Muon_pt'],
    'eta':  arrays['Muon_eta'],
    'phi':  arrays['Muon_phi'],
    'mass': arrays['Muon_mass'],
}, with_name='Momentum4D')

dimuon_mass = (muons[:, 0] + muons[:, 1]).mass

# Plot dimuon mass
h = hist.Hist(hist.axis.Regular(50, 100, 150, label=r'$m_{\mu\mu}$ [GeV]'))
h.fill(dimuon_mass)

h.plot()
plt.title('Dimuon Invariant Mass')
plt.ylabel('Events')

plt.tight_layout()
plt.show()